In [ ]:
import os
import json
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Config

In [ ]:
SEED = 42

TRAIN_PATH = "../data/train.json"
VAL_PATH = "../data/val.json"
TEST_PATH = "../data/test.json"

VOCAB_SAVE_PATH = "../experiments/saved_models/cnn_vocab.json"
MODEL_SAVE_PATH = "../experiments/saved_models/cnn_lm.pt"
METRICS_SAVE_PATH = "../results/tables/cnn_metrics.json"

BATCH_SIZE = 16
EMB_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 3
DROPOUT = 0.1
LR = 1e-3
EPOCHS = 3
MAX_LEN = 20
MIN_FREQ = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

os.makedirs("../experiments/saved_models", exist_ok=True)
os.makedirs("../results/tables", exist_ok=True)

## Load data

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

train_df = train_df.sample(n=min(10000, len(train_df)), random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(n=min(2000, len(val_df)), random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(n=min(2000, len(test_df)), random_state=SEED).reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

## Build vocabulary

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

def build_vocab(token_lists, min_freq=5):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    vocab = SPECIAL_TOKENS.copy()
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab.append(token)

    stoi = {token: idx for idx, token in enumerate(vocab)}
    itos = {idx: token for token, idx in stoi.items()}
    return vocab, stoi, itos, counter

vocab, stoi, itos, counter = build_vocab(train_df["tokens"], min_freq=MIN_FREQ)

print("Vocab size:", len(vocab))
print(counter.most_common(20))

In [ ]:
with open(VOCAB_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(stoi, f, ensure_ascii=False, indent=2)

## Dataset and DataLoader

In [ ]:
PAD_ID = stoi["<pad>"]
UNK_ID = stoi["<unk>"]
BOS_ID = stoi["<bos>"]
EOS_ID = stoi["<eos>"]

def encode_tokens(tokens, stoi, max_len=20):
    token_ids = [stoi.get(tok, UNK_ID) for tok in tokens]
    token_ids = token_ids[: max_len - 2]

    input_ids = [BOS_ID] + token_ids
    target_ids = token_ids + [EOS_ID]

    return input_ids, target_ids

In [ ]:
class CodeSwitchLMDataset(Dataset):
    def __init__(self, df, stoi, max_len=20):
        self.samples = []

        for _, row in df.iterrows():
            tokens = row["tokens"]
            switches = row["switches"]

            input_ids, target_ids = encode_tokens(tokens, stoi, max_len=max_len)

            real_token_count = len(target_ids) - 1
            switches = switches[: max(0, real_token_count - 1)]

            self.samples.append({
                "input_ids": input_ids,
                "target_ids": target_ids,
                "tokens": tokens[:max_len-2],
                "switches": switches
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
def collate_fn(batch):
    max_input_len = max(len(item["input_ids"]) for item in batch)
    max_target_len = max(len(item["target_ids"]) for item in batch)

    batch_input_ids = []
    batch_target_ids = []
    batch_switches = []

    for item in batch:
        input_ids = item["input_ids"]
        target_ids = item["target_ids"]

        input_pad = input_ids + [PAD_ID] * (max_input_len - len(input_ids))
        target_pad = target_ids + [PAD_ID] * (max_target_len - len(target_ids))

        batch_input_ids.append(input_pad)
        batch_target_ids.append(target_pad)
        batch_switches.append(item["switches"])

    return {
        "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
        "target_ids": torch.tensor(batch_target_ids, dtype=torch.long),
        "switches": batch_switches
    }

In [ ]:
train_dataset = CodeSwitchLMDataset(train_df, stoi, max_len=MAX_LEN)
val_dataset = CodeSwitchLMDataset(val_df, stoi, max_len=MAX_LEN)
test_dataset = CodeSwitchLMDataset(test_df, stoi, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

## CNN Language Model

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=self.padding,
            dilation=dilation
        )

    def forward(self, x):
        out = self.conv(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        return out

In [ ]:
class CNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers=3, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)

        layers = []
        in_channels = emb_dim

        for i in range(num_layers):
            dilation = 2 ** i
            layers.append(CausalConv1d(in_channels, hidden_dim, kernel_size=3, dilation=dilation))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_channels = hidden_dim

        self.cnn = nn.Sequential(*layers)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        x = self.embedding(input_ids)      # [B, T, E]
        x = x.transpose(1, 2)              # [B, E, T]
        x = self.cnn(x)                    # [B, H, T]
        x = x.transpose(1, 2)              # [B, T, H]
        logits = self.fc(x)                # [B, T, V]
        return logits

In [ ]:
model = CNNLanguageModel(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model

## Evaluation helpers

In [ ]:
def perplexity_from_loss(loss):
    return math.exp(loss)

In [ ]:
def token_lang_from_id(token_id, itos):
    token = itos.get(token_id, "<unk>")
    for ch in token:
        if "а" <= ch <= "я" or "А" <= ch <= "Я":
            return "RU"
    return "EN"

In [ ]:
def evaluate_model(model, data_loader, criterion, device, itos):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    switch_correct = 0
    switch_total = 0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            target_ids = batch["target_ids"].to(device)

            logits = model(input_ids)
            vocab_size = logits.size(-1)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                target_ids.reshape(-1)
            )

            total_loss += loss.item()
            total_batches += 1

            preds = logits.argmax(dim=-1).cpu().numpy()
            gold_input = batch["input_ids"].cpu().numpy()

            for i in range(len(batch["switches"])):
                gold_switches = batch["switches"][i]

                for j, gold_switch in enumerate(gold_switches):
                    current_token_id = gold_input[i][j + 1]
                    pred_next_token_id = preds[i][j + 1]

                    current_lang = token_lang_from_id(int(current_token_id), itos)
                    pred_next_lang = token_lang_from_id(int(pred_next_token_id), itos)

                    pred_switch = 1 if current_lang != pred_next_lang else 0

                    switch_correct += int(pred_switch == gold_switch)
                    switch_total += 1

    avg_loss = total_loss / max(1, total_batches)

    return {
        "loss": avg_loss,
        "perplexity": perplexity_from_loss(avg_loss),
        "switch_accuracy": switch_correct / max(1, switch_total)
    }

## Training

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(data_loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(device)
        target_ids = batch["target_ids"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        vocab_size = logits.size(-1)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            target_ids.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(1, total_batches)

In [ ]:
best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, criterion, DEVICE, itos)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_perplexity": perplexity_from_loss(train_loss),
        "val_loss": val_metrics["loss"],
        "val_perplexity": val_metrics["perplexity"],
        "val_switch_accuracy": val_metrics["switch_accuracy"]
    }

    history.append(row)

    print(f"Epoch {epoch}")
    print(f"  train_loss: {train_loss:.4f}")
    print(f"  train_ppl:  {perplexity_from_loss(train_loss):.4f}")
    print(f"  val_loss:   {val_metrics['loss']:.4f}")
    print(f"  val_ppl:    {val_metrics['perplexity']:.4f}")
    print(f"  val_switch_acc: {val_metrics['switch_accuracy']:.4f}")

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("  Saved best model!")

## Save results

In [ ]:
history_df = pd.DataFrame(history)
history_df

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
test_metrics = evaluate_model(model, test_loader, criterion, DEVICE, itos)
test_metrics

In [ ]:
final_metrics = {
    "model": "CNN_LM",
    "vocab_size": len(vocab),
    "embedding_dim": EMB_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "epochs": EPOCHS,
    "best_val_loss": best_val_loss,
    "test_loss": test_metrics["loss"],
    "test_perplexity": test_metrics["perplexity"],
    "test_switch_accuracy": test_metrics["switch_accuracy"]
}

with open(METRICS_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(final_metrics, ensure_ascii=False, indent=2))

In [ ]:
history_df.to_csv("../results/tables/cnn_training_history.csv", index=False)
print("Saved CNN training history.")